# Finding public cancer datasets in METASPACE

**METASPACE in Practice: Cancer Research with Public Data** · Notebook 1

The METASPACE knowledgebase holds thousands of public spatial metabolomics datasets. In this notebook we use the Python client to:

1. search the knowledgebase for cancer-related datasets using keywords,
2. look up the projects each dataset belongs to, and the publication DOI where there is one,
3. collect everything in one table that you can filter, save and share.

Install the client once with `pip install metaspace2020`. No login is needed for public data.

In [1]:
from pathlib import Path

import pandas as pd

from metaspace import SMInstance

sm = SMInstance()  # connects to https://metaspace2020.org, no API key needed for public data
sm

SMInstance(https://metaspace2020.org/graphql)

## 1. Search by keyword

`sm.datasets()` takes search filters. Two are useful for finding disease-related data:

- `condition=` matches the **Condition** field filled in at submission (for example "Cancer" or "Glioblastoma"),
- `nameMask=` matches the **dataset name**.

Both are substring matches that ignore case. The Condition field is free text, so authors spell things in different ways. We therefore search for a list of keywords. Edit the list to widen or narrow the search.

In [2]:
KEYWORDS = [
    "cancer", "tumor", "tumour", "carcinoma", "adenocarcinoma", "neoplasm", "malignant",
    "metastasis", "glioma", "glioblastoma", "melanoma", "lymphoma", "leukemia", "sarcoma",
]

hits = {}  # dataset id -> (dataset, set of "keyword (field)" matches)
for keyword in KEYWORDS:
    for field, search in [("condition", {"condition": keyword}), ("name", {"nameMask": keyword})]:
        for ds in sm.datasets(**search):
            hits.setdefault(ds.id, (ds, set()))[1].add(f"{keyword} ({field})")

print(f"{len(hits)} unique datasets found for {len(KEYWORDS)} keywords")

809 unique datasets found for 14 keywords


## 2. Projects and publication DOIs

A dataset can belong to one or more **projects**. A project of a published study carries the publication's **DOI** as an external link. We download all projects once and look them up by ID.

In [3]:
projects = {p["id"]: p for p in sm.projects.get_all_projects()}


def project_doi(project):
    links = project.get("externalLinks") or []
    return "; ".join(link["link"] for link in links if link["provider"] == "DOI")


print(f"{len(projects)} projects, {sum(bool(project_doi(p)) for p in projects.values())} with a DOI")

540 projects, 112 with a DOI


## 3. Build the table

One row per dataset. The `url` opens the dataset's annotations in the METASPACE web app.

In [4]:
def get(metadata, *path):
    for key in path:
        metadata = (metadata or {}).get(key)
    return metadata


rows = []
for ds_id, (ds, matches) in hits.items():
    md_ = ds.metadata
    ds_projects = [projects.get(p["id"], p) for p in ds.projects]
    rows.append({
        "dataset_id": ds_id,
        "name": ds.name,
        "url": f"https://metaspace2020.org/annotations?ds={ds_id}",
        "condition": get(md_, "Sample_Information", "Condition"),
        "organism": get(md_, "Sample_Information", "Organism"),
        "organism_part": get(md_, "Sample_Information", "Organism_Part"),
        "ionisation_source": get(md_, "MS_Analysis", "Ionisation_Source"),
        "analyzer": get(md_, "MS_Analysis", "Analyzer"),
        "polarity": ds.polarity,
        "matched": "; ".join(sorted(matches)),
        "project_ids": "; ".join(p["id"] for p in ds_projects),
        "project_names": "; ".join(p["name"] for p in ds_projects),
        "publication_status": "; ".join(p["publicationStatus"] for p in ds_projects),
        "doi": "; ".join(d for d in (project_doi(p) for p in ds_projects) if d),
    })

cancer_datasets = pd.DataFrame(rows).sort_values("name").reset_index(drop=True)
Path("data").mkdir(exist_ok=True)
cancer_datasets.to_csv("data/cancer_datasets.csv", index=False)
cancer_datasets.shape

(809, 14)

In [5]:
pd.set_option("display.max_colwidth", 60)
cancer_datasets[["dataset_id", "name", "organism_part", "project_ids", "doi", "url"]].head(10)

,dataset_id,name,organism_part,project_ids,doi,url
0,2026-06-19_19h35m00s,041025_tumorslide1_dan_bo_slicec_neg_timsoff-total ion c...,Whole organism,,,https://metaspace2020.org/annotations?ds=2026-06-19_19h3...
1,2026-06-19_19h16m12s,041025_tumorslide1_dan_bo_slicec_neg_timsoff-total ion c...,Whole organism,,,https://metaspace2020.org/annotations?ds=2026-06-19_19h1...
2,2026-06-12_21h55m47s,041025_tumorslide1_dan_bogfp_slicec_neg_timsoff-total io...,Whole organism,,,https://metaspace2020.org/annotations?ds=2026-06-12_21h5...
3,2026-04-28_22h13m15s,04242026_neg_1_05,ovaries,,,https://metaspace2020.org/annotations?ds=2026-04-28_22h1...
4,2022-05-07_01h01m45s,05042022_torti_c12_c13_v2,Ovary,92686fb8-9ea0-11ec-89bf-8bec110f5427; 6822182c-00b8-11ea...,,https://metaspace2020.org/annotations?ds=2022-05-07_01h0...
5,2022-05-06_22h55m06s,05042022_torti_c12_c13_v2_erastin,Ovary,92686fb8-9ea0-11ec-89bf-8bec110f5427; 6822182c-00b8-11ea...,,https://metaspace2020.org/annotations?ds=2022-05-06_22h5...
6,2018-06-05_17h32m02s,05062018_AI_Spheroid3_DAN_Neg_p50xp50,Colon,,,https://metaspace2020.org/annotations?ds=2018-06-05_17h3...
7,2018-07-05_12h35m48s,05072018_AR_Rho5002D_DHB_pos_p50_s50,Cervix,,,https://metaspace2020.org/annotations?ds=2018-07-05_12h3...
8,2018-07-05_19h28m34s,05072018_AR_Rho5002D_DHB_pos_p50_s50_recal,Cervix,,,https://metaspace2020.org/annotations?ds=2018-07-05_19h2...
9,2019-10-05_21h34m29s,051316_GSH_QMSI_CancerTissue,Ovary,,,https://metaspace2020.org/annotations?ds=2019-10-05_21h3...


## 4. Explore the table

Which datasets belong to a published study? Keep the ones whose project has a DOI.

In [6]:
published = cancer_datasets[cancer_datasets["doi"] != ""]
print(f"{len(published)} of {len(cancer_datasets)} datasets are linked to a project with a DOI")
published[["dataset_id", "name", "organism_part", "doi"]].head(10)

133 of 809 datasets are linked to a project with a DOI


,dataset_id,name,organism_part,doi
88,2020-11-29_20h32m07s,1999_129T_NEG,Ovary,https://doi.org/10.1007/s11306-021-01819-x
89,2020-11-29_20h27m30s,1999_129T_POS,Ovary,https://doi.org/10.1007/s11306-021-01819-x
90,2021-05-04_21h13m55s,2000-117Tumor_NEG,Bladder,https://doi.org/10.1007/s11306-021-01819-x
91,2021-05-04_21h17m03s,2000-117Tumor_POS,Bladder,https://doi.org/10.1007/s11306-021-01819-x
92,2021-05-04_20h41m28s,2002-322Tumor_NEG,Bladder,https://doi.org/10.1007/s11306-021-01819-x
93,2021-05-04_20h32m54s,2002-322Tumor_POS,Bladder,https://doi.org/10.1007/s11306-021-01819-x
94,2021-05-04_20h47m58s,2002-342Tumor-NEG,Bladder,https://doi.org/10.1007/s11306-021-01819-x
95,2021-05-04_20h51m33s,2002-342Tumor-POS,Bladder,https://doi.org/10.1007/s11306-021-01819-x
96,2021-05-04_21h21m03s,2003-192Tumor-POS,Bladder,https://doi.org/10.1007/s11306-021-01819-x
97,2021-05-04_21h19m54s,2003-192Tumor_NEG,Bladder,https://doi.org/10.1007/s11306-021-01819-x


Which tissues are covered, and how were the datasets measured?

In [7]:
cancer_datasets["organism_part"].value_counts().head(10)

organism_part
Breast          140
Ovary           136
Lung             97
Brain            82
Kidney           64
Skin             42
Colon            32
Stomach          28
Breast tumor     25
Liver            23
Name: count, dtype: int64

In [8]:
cancer_datasets["ionisation_source"].value_counts().head(8)

ionisation_source
MALDI            589
DESI              80
AP-MALDI          44
AP-SMALDI5        28
AP-SMALDI5 AF     18
IRMALDESI         15
AFADESI           10
AP-SMALDI 5        8
Name: count, dtype: int64

The keyword search is deliberately broad, so check the results before using them: a name that contains "tumor" can also describe a control, for example. Use the `matched` column to see which keyword found a dataset, and open the `url` to inspect it in the web app.

**Next:** notebook 2 takes a published cancer project and uses the METASPACE converter to bring its datasets into AnnData.